# F580.2 轉骰子：題解課件
Status: in_use  
Prereq: input()/split()、list/tuple、對應題型的基礎語法  
Can-do checklist:  
- [ ] 能用狀態表示骰子（上/前/右）
- [ ] 能正確實作旋轉與交換操作
- [ ] 能處理輸入 m 次操作並輸出結果


## 一、必要知識點（先備）
1. **模擬（simulation）**：依規則逐步更新狀態。
2. **狀態表示**：一個物件用多個變數/結構表示其狀態。
3. **陣列/列表**：用 list 存六個面，旋轉就是重新對應。
4. **索引與交換**：1-based 題號 → 0-based 索引。

Python 內建 swap 寫法：`a, b = b, a`。


## 結構化打印（方便觀察骰子六面）
用固定格式輸出 top/bottom/front/back/right/left，方便檢查旋轉是否正確。


In [50]:
def print_dice(dice):
    labels = ["top", "bottom", "front", "back", "right", "left"]
    print("-" * 32)
    for label, value in zip(labels, dice):
        print(f"{label:<6}: {value}")
    print("-" * 32)

# demo
dice = [1, 6, 4, 3, 2, 5]
print_dice(dice)


--------------------------------
top   : 1
bottom: 6
front : 4
back  : 3
right : 2
left  : 5
--------------------------------


### Swap 小練習
把下列程式補完整，交換兩個變數：


In [51]:
x = 5
y = 9
# TODO: 使用 swap 交換 x 與 y
# x, y = y, x
print(x, y)


5 9



以下概念要先講清楚，後面解題才不會卡住。

### 1) 模擬（simulation）
把每一步操作當成規則，逐步更新狀態。

**重點：** 不要想公式，照規則做就對了。



In [52]:
# demo: ops and update must be defined before the loop
def update(state, op):
    return state + op

ops = [1, -1, 2]
state = 0
for op in ops:
    state = update(state, op)
state


2



### 2) 狀態表示（state representation）
一個物件的狀態可以用多個變數/結構描述。
本題一顆骰子的狀態 = 六個面的點數排列。



In [53]:
# 固定順序：top, bottom, front, back, right, left
dice = [1, 6, 4, 3, 2, 5]




### 3) 陣列/列表與交換
多顆骰子可以用 list 存起來，交換時直接 swap 整顆骰子。



In [54]:
# demo: define n and a,b before using
n = 3
a, b = 1, 2
dices = [dice[:] for _ in range(n)]
# 交換第 a 顆與第 b 顆
dices[a-1], dices[b-1] = dices[b-1], dices[a-1]
dices


[[1, 6, 4, 3, 2, 5], [1, 6, 4, 3, 2, 5], [1, 6, 4, 3, 2, 5]]



### 4) 索引（1-based vs 0-based）
題目編號從 1 開始，但 Python list 從 0 開始。



In [55]:
idx = a - 1  # 轉成 0-based


## 二、資料結構設計
### 1) 單顆骰子的表示
固定順序保存六個面：
```
index: 0    1      2      3     4     5
face : top bottom front back right left
```
初始：top=1, front=4, right=2 → bottom=6, back=3, left=5

例：
```
dice = [1, 6, 4, 3, 2, 5]
```

### 2) 多顆骰子的表示
```
dices = [dice1, dice2, ..., diceN]
```
交換時直接 swap list 的元素即可。


## 三、核心操作：旋轉規則
**向前旋轉（b = -1）**
- top → front
- front → bottom
- bottom → back
- back → top
- left/right 不變

**向右旋轉（b = -2）**
- top → right
- right → bottom
- bottom → left
- left → top
- front/back 不變


### 旋轉小範例
用初始骰子做一次「向前」與一次「向右」的結果。


In [56]:
# top, bottom, front, back, right, left
dice = [1, 6, 4, 3, 2, 5]

def roll_forward(d):
    top, bottom, front, back, right, left = d
    return [back, front, top, bottom, right, left]

def roll_right(d):
    top, bottom, front, back, right, left = d
    return [left, right, front, back, top, bottom]

print("start :", dice)
print("forward:", roll_forward(dice))
print("right :", roll_right(dice))


start : [1, 6, 4, 3, 2, 5]
forward: [3, 4, 1, 6, 2, 5]
right : [5, 2, 4, 3, 1, 6]


## 方向與座標軸的釐清（避免誤解）
題目雖然只說「向前轉 / 向右轉」，但其實**方向已被初始朝向固定**：

- 點數 **1 朝上**（top）
- 點數 **4 朝前**（front）
- 點數 **2 朝右**（right）

因此我們可以把它視為固定的三軸座標：
- 上/下軸：top ↔ bottom
- 前/後軸：front ↔ back
- 右/左軸：right ↔ left

**向前轉**就是沿「左右軸」翻滾：
- top → front → bottom → back → top

**向右轉**就是沿「前後軸」翻滾：
- top → right → bottom → left → top

只要用這個固定座標系，題目就不會有歧義。


## 笨笨但穩：最直覺的暴力思路
如果你只想先寫出能過的版本，不追求優雅，可以用最直觀的方式：

1. **每顆骰子都用 6 個面直接存**（top, bottom, front, back, right, left）。
2. **每次操作就直接改那顆骰子的 6 個值**。
3. **交換就直接交換整顆骰子 list**。

這個方法沒有任何技巧：
- 不做數學推導
- 不做壓縮表示
- 不做進階資料結構

但它非常安全，因為每一步都和題目規則一一對應。


### 暴力版偽碼
```
init dice = [1,6,4,3,2,5]
dices = [copy(dice) for i in 1..n]

for each (a,b):
    if b > 0:
        swap(dices[a-1], dices[b-1])
    else if b == -1:
        roll_forward(dices[a-1])
    else if b == -2:
        roll_right(dices[a-1])

print top face of all dices
```


In [57]:
# 旋轉函式
def roll_forward(dice):
    top, bottom, front, back, right, left = dice
    dice[0] = back
    dice[1] = front
    dice[2] = top
    dice[3] = bottom
    # right, left 不變

def roll_right(dice):
    top, bottom, front, back, right, left = dice
    dice[0] = left
    dice[1] = right
    dice[4] = top
    dice[5] = bottom
    # front, back 不變


## 四、解題流程（高層邏輯）
1. 讀入 n, m
2. 初始化 n 顆骰子（同一初始狀態）
3. 依序處理 m 次操作：
   - a, b > 0 → 交換 a 與 b
   - b = -1 → 第 a 顆向前旋轉
   - b = -2 → 第 a 顆向右旋轉
4. 輸出每顆骰子 top 面


In [58]:
def solve(ops, n):
    base = [1, 6, 4, 3, 2, 5]
    dices = [base[:] for _ in range(n)]

    for a, b in ops:
        if b > 0:
            dices[a-1], dices[b-1] = dices[b-1], dices[a-1]
        elif b == -1:
            roll_forward(dices[a-1])
        elif b == -2:
            roll_right(dices[a-1])

    return [dice[0] for dice in dices]


## 五、範例驗證
範例 1：
``
n=1, m=2
1 -2
1 -1
``


In [59]:
print(solve([(1, -2), (1, -1)], n=1))  # 預期輸出: [3]


[3]


範例 2：
``
n=3, m=3
2 -1
3 -2
3 1
``


In [60]:
print(solve([(2, -1), (3, -2), (3, 1)], n=3))  # 預期輸出: [5, 3, 1]


[5, 3, 1]


## 六、易錯點提醒
1. 題目骰子編號是 1-based。
2. 旋轉規則方向不可弄反。
3. 交換的是整顆骰子狀態。


## 七、延伸問題
1. 如果加入向左/向後旋轉，規則如何寫？
2. 若骰子初始狀態不同，如何調整輸入與初始化？
3. 能否用矩陣/向量方法來描述骰子旋轉？


## 八、另一種更適合的資料結構：命名欄位 (NamedTuple)
目的：保持可讀性，同時避免索引記錯。

- 優點：`d.top` 比 `d[0]` 更清楚
- 旋轉時只做欄位重排


In [61]:
from typing import NamedTuple

class Dice(NamedTuple):
    top: int
    bottom: int
    front: int
    back: int
    right: int
    left: int

def roll_forward(d: Dice) -> Dice:
    # top -> front -> bottom -> back -> top
    return Dice(d.back, d.front, d.top, d.bottom, d.right, d.left)

def roll_right(d: Dice) -> Dice:
    # top -> right -> bottom -> left -> top
    return Dice(d.left, d.right, d.front, d.back, d.top, d.bottom)

def swap_dice(dices, a, b):
    dices[a], dices[b] = dices[b], dices[a]

d = Dice(1, 6, 4, 3, 2, 5)
print(roll_forward(d))
print(roll_right(d))


Dice(top=3, bottom=4, front=1, back=6, right=2, left=5)
Dice(top=5, bottom=2, front=4, back=3, right=1, left=6)


### 解題步驟範例（NamedTuple 版本）
用一組小操作示範「交換 + 旋轉」的流程與結果。


In [62]:
# ops: (a, b)  b>0 表示交換；b=-1 向前；b=-2 向右
ops = [(1, -1), (2, -2), (1, 2)]
n = 2

dices = [Dice(1, 6, 4, 3, 2, 5) for _ in range(n)]

for a, b in ops:
    idx = a - 1
    if b > 0:
        jdx = b - 1
        dices[idx], dices[jdx] = dices[jdx], dices[idx]
    elif b == -1:
        dices[idx] = roll_forward(dices[idx])
    elif b == -2:
        dices[idx] = roll_right(dices[idx])

# 輸出每顆骰子的 top 面
[d.top for d in dices]


[5, 3]